<a href="https://colab.research.google.com/github/ganesh7207/AI-ML-Projects/blob/main/VOICE_TRANSLATOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q "transformers<5.0.0" gradio sentencepiece gTTS librosa

In [2]:

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# Telugu speech recognition
print("Loading Telugu speech recognition model...")
asr = pipeline(
    "automatic-speech-recognition",
    model="vanshnawander/whisper-small-telugu",
    device=0 if torch.cuda.is_available() else -1
)

# Telugu-to-English translation
print("Loading translation model...")
TRANSLATION_MODEL = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(
    TRANSLATION_MODEL,
    src_lang="tel_Telu"
)

translator = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL
)

device = "cuda" if torch.cuda.is_available() else "cpu"
translator = translator.to(device)

print("All models are ready!")


Loading Telugu speech recognition model...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

All models are ready!


In [3]:

from gtts import gTTS
import gradio as gr
import tempfile
import os

def voice_translate(audio_path):
    if audio_path is None:
        return "", "", None

    # 1. Telugu speech to text
    speech_result = asr(
        audio_path,
        generate_kwargs={"language": "te", "task": "transcribe"}
    )
    telugu_text = speech_result["text"].strip()

    if not telugu_text:
        return "", "No speech detected. Please try again.", None

    # 2. Telugu text to English
    inputs = tokenizer(
        telugu_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = translator.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),
            max_new_tokens=200
        )

    english_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # 3. English text to voice
    output_path = tempfile.NamedTemporaryFile(
        suffix=".mp3", delete=False
    ).name

    gTTS(text=english_text, lang="en").save(output_path)

    return telugu_text, english_text, output_path


In [4]:

app = gr.Interface(
    fn=voice_translate,
    inputs=gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="🎙️ Speak in Telugu"
    ),
    outputs=[
        gr.Textbox(label="Telugu Speech Recognized", lines=3),
        gr.Textbox(label="English Translation", lines=3),
        gr.Audio(label="🔊 English Voice Output", type="filepath")
    ],
    title="Telugu to English Voice Translator",
    description=(
        "Record Telugu speech, translate it into English, "
        "and listen to the English voice output."
    )
)

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b49ce5b959d587aede.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
